# BirdSense — Colab Training Notebook

**Before running:**
1. Set runtime type to **GPU** — Runtime > Change runtime type > T4 GPU
2. Upload your project to Google Drive so the layout below matches
3. Run all cells top-to-bottom

**Expected Drive layout:**
```
MyDrive/
└── birdsense/
    ├── dataset/
    │   ├── processed/      <- upload this (8475 wav clips)
    │   └── test_holdout/   <- upload this (holdout mp3s)
    ├── scripts/            <- upload all .py files
    ├── species_selected.csv
    └── requirements_train.txt
```

## Step 1 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Copy Project to Local Disk

Edit `DRIVE_DIR` if your Drive folder is named differently.

Training reads thousands of small files per epoch — the Drive FUSE mount is far slower than local disk for this access pattern, so we copy the dataset and scripts to `/content` once per session and work from there instead.

In [ ]:
import os
import shutil

DRIVE_DIR = '/content/drive/MyDrive/birdsense'
PROJECT_DIR = '/content/birdsense'

if not os.path.exists(PROJECT_DIR):
    print('Copying project to local disk (this can take a few minutes)...')
    shutil.copytree(DRIVE_DIR, PROJECT_DIR)
    print('Copy complete.')
else:
    print('Local copy already exists, skipping copy.')

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

for folder in ['dataset/processed', 'dataset/test_holdout', 'scripts']:
    status = 'OK' if os.path.isdir(folder) else 'MISSING'
    print(f'  [{status}]  {folder}')

## Step 3 — Install Missing Dependencies

Colab already ships with TensorFlow, scikit-learn, and requests.
We only install what is actually missing to avoid version conflicts.

In [5]:
# Do NOT reinstall tensorflow — Colab already has it and reinstalling causes conflicts
!pip install -q "librosa>=0.10" "soundfile>=0.12" "birdnetlib>=0.18"

## Step 4 — Check GPU

In [6]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {gpus}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.')

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Step 5 — Check Dataset

Verify clip counts per species before training.

In [7]:
from pathlib import Path

processed = Path('dataset/processed')
holdout = Path('dataset/test_holdout')

total_clips = 0
print(f'{"Species":<35} {"Processed":>10} {"Holdout":>10}')
print('-' * 57)
for species_dir in sorted(processed.iterdir()):
    if not species_dir.is_dir():
        continue
    clips = len(list(species_dir.glob('*.wav')))
    holdout_dir = holdout / species_dir.name
    ho = len(list(holdout_dir.glob('*.mp3'))) if holdout_dir.exists() else 0
    total_clips += clips
    print(f'{species_dir.name:<35} {clips:>10} {ho:>10}')
print('-' * 57)
print(f'{"TOTAL":<35} {total_clips:>10}')

Species                              Processed    Holdout
---------------------------------------------------------
black-crowned_pitta                        548          9
blue-eared_barbet                         1043         28
blue-headed_pitta                          297          6
bold-striped_tit-babbler                   387         11
bornean_banded_pitta                       189          5
bornean_black_magpie                       364         13
bornean_bristlehead                        157          5
bornean_ground_cuckoo                      218          6
bornean_treepie                            212          7
golden-naped_barbet                        306          8
greater_racket-tailed_drongo              2418         54
rhinoceros_hornbill                        418         10
rufous-crowned_babbler                     675         15
white-chested_babbler                      797         19
white-crowned_shama                        446         11
--------------

## Step 6 — Extract BirdNET Embeddings

Runs every processed clip through the pretrained BirdNET model (frozen — no training here) and caches a 1024-d embedding per clip to `models/checkpoints/embeddings_train.npz`. This is the slow part (one forward pass per clip); train.py then reads the cache and trains in seconds.

In [ ]:
!python scripts/extract_embeddings.py

## Step 7 — Train

- Loads cached BirdNET embeddings from Step 6
- Trains a small Dense classification head (BirdNET itself stays frozen)
- Runs for up to 60 epochs with early stopping
- Saves best checkpoint to `models/checkpoints/best_model.keras`

In [ ]:
!python scripts/train.py

BirdSense Trainer — EfficientNetB0 end-to-end fine-tuning
TensorFlow 2.20.0  |  GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Species: 15  |  Total clips: 8475
Train: 6949  |  Val: 1526
Class weight range: 0.234 — 3.591

2026-07-04 04:55:53.062503: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1783140953.063949    3825 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Model: "birdsense"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spectrogram 

^C


## Step 8 — Evaluate

Runs the trained model on the held-out test set.
Checks macro-F1 against the 0.80 export gate.
Report saved to `models/checkpoints/eval_report.txt`.

In [ ]:
!python scripts/evaluate.py

## Step 9 — Export

**Only run if Step 8 printed PASS.**

Saves:
- `models/export/model.tflite`
- `models/export/labels.txt`

In [ ]:
!python scripts/export.py

## Step 10 — Download Outputs

Downloads the exported files to your local machine.

In [ ]:
from google.colab import files
import os

outputs = [
    'models/export/model.tflite',
    'models/export/labels.txt',
    'models/checkpoints/eval_report.txt',
    'models/checkpoints/label_map.json',
]

for path in outputs:
    if os.path.exists(path):
        print(f'Downloading: {path}')
        files.download(path)
    else:
        print(f'Not found (skipping): {path}')